# 이온 밀 식각 장비 예지보전 EDA v3 — 1분 요약 기준

| 항목 | 내용 |
|---|---|
| 서비스 목표 | 웨이퍼 뒷면 헬륨 냉각(flowcool) 계통 고장 3종을 **고장 전에 경고**해 정비를 앞당긴다 |
| 분석 대상 | 학습 장비 20대 전체. 4초 원본을 **1분 단위로 요약**한 5,777,042행 (원본 학습 82,189,440행의 7.0%, **기간은 100%**) |
| 입력 후보 | 연속형 센서 16개의 1분 평균 (표준화 값). 셔터·레시피는 조건 구분용 |
| 기간 | 장비별 약 338~489일 (`time`을 초로 가정). 실제 날짜 없음 |
| 비교 방식 | 같은 장비·레시피·단계·셔터 조건의 **평소 구간 대비 z값**으로, 고장 24시간 전과 평소를 비교 |
| 완료 범위 | 품질·간격, 장비 간 차이, 상관, 고장 전 관측 가능성, 센서별 구분력, 고장 전 궤적, 사건 묶기 민감도 |
| 미수행 | 머신러닝 학습·성능 평가. 시험 장비 5대(정답 미공개) 분석. 4초 해상도의 순간 변화 분석 |

원본을 일부 행만 뽑은 표본 검사가 아니다. 20대 전체 기간을 1분 평균·최소·최대로 줄인 파일(`data/processed/minute/`)을 읽는다. 이 노트북은 **그림·표를 파일로 저장하지 않고** 노트북 안에서만 보여준다.

## 1. 분석 목적

| 질문 | 이 답으로 정할 것 |
|---|---|
| Q1. 어떤 센서가 고장 직전과 평소를 구분하는가? | 입력 센서 후보 |
| Q2. 고장 징후는 고장 몇 시간 전부터 나타나는가? | 예측 구간(몇 시간 앞을 볼지)과 입력 길이 |
| Q3. 장비마다 고장 횟수와 센서 수준이 얼마나 다른가? | 장비 단위 평가·장비별 기준 필요성 |
| Q4. 고장 직전에 관측이 충분한가? 고장 기록은 몇 건의 사건인가? | 라벨 규칙과 학습 가능한 사건 수 |
| Q5. 센서끼리 중복되는가? | 피처 선택·차원 축소 필요성 |

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import rankdata
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
ROOT = next(p for p in (cwd, cwd.parent) if (p / 'data/processed/minute').exists())
PROC = ROOT / 'data/processed'

def show(fig, height=520):
    """그림은 파일로 저장하지 않고 노트북에만 표시한다."""
    fig.update_layout(template='plotly_white', height=height, font=dict(family='Malgun Gothic, sans-serif'),
                      margin=dict(l=30, r=30, t=75, b=70))
    display({'application/vnd.plotly.v1+json': json.loads(fig.to_json())}, raw=True)

SENSORS = ['IONGAUGEPRESSURE', 'ETCHBEAMVOLTAGE', 'ETCHBEAMCURRENT', 'ETCHSUPPRESSORVOLTAGE', 'ETCHSUPPRESSORCURRENT',
           'FLOWCOOLFLOWRATE', 'FLOWCOOLPRESSURE', 'ETCHGASCHANNEL1READBACK', 'ETCHPBNGASREADBACK',
           'FIXTURETILTANGLE', 'ROTATIONSPEED', 'ACTUALROTATIONANGLE', 'ETCHSOURCEUSAGE', 'ETCHAUXSOURCETIMER',
           'ETCHAUX2SOURCETIMER', 'ACTUALSTEPDURATION']
NAMES = {'IONGAUGEPRESSURE': '챔버 압력', 'ETCHBEAMVOLTAGE': '빔 전압', 'ETCHBEAMCURRENT': '빔 전류',
         'ETCHSUPPRESSORVOLTAGE': '억제 전압', 'ETCHSUPPRESSORCURRENT': '억제 전류', 'FLOWCOOLFLOWRATE': '냉각 유량',
         'FLOWCOOLPRESSURE': '냉각 압력', 'ETCHGASCHANNEL1READBACK': '아르곤 유량(소스)', 'ETCHPBNGASREADBACK': '아르곤 유량(PBN)',
         'FIXTURETILTANGLE': '웨이퍼 기울기', 'ROTATIONSPEED': '회전 속도', 'ACTUALROTATIONANGLE': '회전 각도',
         'ETCHSOURCEUSAGE': '그리드 사용량', 'ETCHAUXSOURCETIMER': '실드 사용량1', 'ETCHAUX2SOURCETIMER': '실드 사용량2',
         'ACTUALSTEPDURATION': '단계 소요시간'}
FAULTS = ['FlowCool Pressure Dropped Below Limit', 'Flowcool Pressure Too High Check Flowcool Pump', 'Flowcool leak']
FSHORT = dict(zip(FAULTS, ['압력 저하', '압력 과다', '누설']))
TTF = {f: 'TTF_' + f for f in FAULTS}
FEAT = [s + '_mean' for s in SENSORS]
HOUR = 3600
C_FAIL, C_OK = '#eb6834', '#2a78d6'

cols = ['Tool', 'time_start', 'n_rows', 'recipe', 'recipe_step', 'shutter'] + list(TTF.values()) + FEAT
df = pd.concat([pd.read_parquet(f, columns=cols) for f in sorted((PROC / 'minute').glob('*.parquet'))], ignore_index=True)
df['Tool'] = df.Tool.astype('category')
df = df.sort_values(['Tool', 'time_start']).reset_index(drop=True)

faults = pd.read_csv(PROC / 'faults.csv')
faults['fault_type'] = faults.fault_name.str.replace(' [NoWaferID]', '', regex=False)
events = (faults.drop_duplicates(['Tool', 'time', 'fault_type'])
          .sort_values(['Tool', 'fault_type', 'time']).reset_index(drop=True))

manifest = json.loads((PROC / 'manifest.json').read_text(encoding='utf-8'))
assert len(df) == sum(v['minute_rows'] for v in manifest.values()) and df.Tool.nunique() == 20
assert len(faults) == 1236 and len(events) == 1211
print(f'1분 요약 {len(df):,}행 · 장비 {df.Tool.nunique()}대 · 고장 기록 {len(faults):,}행(중복 제외 {len(events):,}건)')

1분 요약 5,777,042행 · 장비 20대 · 고장 기록 1,236행(중복 제외 1,211건)


## 2. 데이터셋 설명

### 2-1. 출처·범위

[PHM Society 2018 Data Challenge](https://www.phmsociety.org/events/conference/phm/18/data-challenge)의 이온 밀 식각 장비 데이터다. 장비 1대가 웨이퍼를 처리하는 동안 약 4초마다 센서·설정값을 기록하고, 헬륨 냉각 계통 고장 3종의 발생 시각(작업자가 정비를 위해 장비를 세운 시점)을 따로 제공한다. 자세한 설명은 [data/README.md](../data/README.md)에 있다.

이 노트북이 읽는 **1분 요약 파일**은 `src/make_processed.py`가 만든다. 같은 분(`time // 60`)에 속한 기록을 묶어 센서별 평균·최소·최대와 TTF를 남겼다. 값까지 같은 중복 행 5,470행만 제거했다. 기록이 없는 분은 행으로 만들지 않았다.

시험 데이터 5대는 정답이 공개되지 않아 쓰지 않는다. 성능 평가는 학습 20대 안에서 장비 단위로 나눠 한다.

In [2]:
span = df.groupby('Tool', observed=True).time_start.agg(['min', 'max'])
fc = (events.pivot_table(index='Tool', columns='fault_type', values='time', aggfunc='size', fill_value=0)
      .reindex(columns=FAULTS).rename(columns=FSHORT))
scale = pd.DataFrame({'1분 행 수': df.groupby('Tool', observed=True).size(),
                      '기간(일)': ((span['max'] - span['min']) / 86400).round(0).astype(int)}).join(fc)
scale['고장 합계'] = fc.sum(axis=1)
display(scale)

fig = make_subplots(rows=1, cols=2, subplot_titles=['장비별 1분 요약 행 수', '장비별 고장 기록 수(중복 제외)'])
fig.add_bar(x=scale.index, y=scale['1분 행 수'], marker_color=C_OK, name='1분 행 수', row=1, col=1)
for f, c in zip(FAULTS, ['#2a78d6', '#eb6834', '#7a7f87']):
    fig.add_bar(x=fc.index, y=fc[FSHORT[f]], name=FSHORT[f], marker_color=c, row=1, col=2)
fig.update_layout(barmode='stack', title='그림 2-1. 장비 규모와 고장 기록 — 장비마다 크게 다르다',
                  legend=dict(orientation='h', y=-.2))
show(fig, 460)

,1분 행 수,기간(일),압력 저하,압력 과다,누설,고장 합계
Tool,,,,,,
01M01,218970,460,13,5,1,19
01M02,345344,423,53,40,16,109
02M01,200758,338,9,7,1,17
02M02,294710,384,5,27,0,32
03M01,243115,444,40,51,0,91
03M02,359327,472,298,49,9,356
04M01,362839,467,7,1,1,9
04M02,301192,446,21,1,15,37
05M01,339279,455,14,7,7,28


### 2-2. 컬럼 설명

원본은 24개 컬럼이고 대회 페이지의 정의를 옮겼다. 연속형 센서 16개는 학습 20대 전체 기준으로 **표준화된 값**이라 실제 단위는 알 수 없고, “평소 대비 높음·낮음”까지만 말할 수 있다.

**핵심 컬럼**

| 컬럼 | 설명 | 이름 |
|---|---|---|
| `FLOWCOOLPRESSURE` | 냉각 회로의 헬륨 압력. 고장 3종의 이름이 모두 이 값과 관련된다 | 냉각 압력 |
| `FLOWCOOLFLOWRATE` | 냉각 회로의 헬륨 유량(MFC 제어). 압력과 같이 봐야 공급 문제인지 누설인지 구분된다 | 냉각 유량 |
| `IONGAUGEPRESSURE` | 진공 상태인 메인 공정 챔버의 압력. 누설이 챔버 쪽으로 번지는지 본다 | 챔버 압력 |

**나머지 센서·설정값 13개 (+ 조건 구분용 컬럼)**

| 구분 | 컬럼 | 설명 |
|---|---|---|
| 빔 | `ETCHBEAMVOLTAGE`, `ETCHBEAMCURRENT` | 빔 플레이트 전압, 그리드에 닿는 이온 전류 |
| 빔 | `ETCHSUPPRESSORVOLTAGE`, `ETCHSUPPRESSORCURRENT` | 억제 플레이트 전압, 억제 플레이트 이온 전류 |
| 가스 | `ETCHGASCHANNEL1READBACK`, `ETCHPBNGASREADBACK` | 이온 소스·입자 빔 중화기(PBN)로 들어가는 아르곤 유량 |
| 웨이퍼 | `FIXTURETILTANGLE`, `ROTATIONSPEED`, `ACTUALROTATIONANGLE` | 웨이퍼 기울기·회전 속도 설정, 측정 회전 각도 |
| 소모품 | `ETCHSOURCEUSAGE`, `ETCHAUXSOURCETIMER`, `ETCHAUX2SOURCETIMER` | 그리드·챔버 실드 소모품 사용량 카운터 |
| 공정 | `ACTUALSTEPDURATION` | 공정 단계에 실제로 걸린 시간 |
| 조건 | `recipe`, `recipe_step`, `FIXTURESHUTTERPOSITION`(→`shutter`) | 장비 설정 묶음, 레시피 단계, 셔터 열림·닫힘 코드(0·1·2·3·255) |

**정답 컬럼**: `TTF_*` 3개는 고장까지 남은 시간이다. **결측은 “정해진 기간 안에 그 고장이 없다”는 뜻**이라 0이나 정상으로 바꾸지 않는다. 고장 시각은 `faults.csv`(고장 종류·장비·시각)에 있다.

1분 요약본은 센서마다 `_mean`·`_min`·`_max`가 있고, 이 노트북은 `_mean`만 쓴다.

## 3. 전처리·데이터 품질

### 3-1. 결측·기록 간격

- 센서 값의 결측과 `recipe_step`·`shutter` 결측을 확인한다. TTF 결측은 오류가 아니다.
- 1분 요약은 기록이 있는 분만 행으로 남기므로, **연속한 두 행의 시각 차이가 60초보다 크면 그 사이에 기록이 없다**는 뜻이다. 장비가 멈췄거나 데이터가 빠진 구간이다.
- 한 분에 기록이 15개 미만이면(`n_rows`) 그 분의 평균은 덜 믿을 만하다.
- 통계적으로 큰 값을 임의로 삭제하지 않는다.

In [3]:
miss = (df.isna().mean() * 100).rename('결측률(%)')
miss = miss[miss > 0].round(3)
display(miss.to_frame().rename_axis('컬럼'))
print('센서 평균 16개 결측:', int(df[FEAT].isna().sum().sum()), '건')

df['gap'] = df.groupby('Tool', observed=True).time_start.diff()
g = df.gap.dropna()
gap_tbl = pd.DataFrame({'구간 수': [(g > 60).sum(), (g > 600).sum(), (g > HOUR).sum(), (g > 86400).sum()]},
                       index=['1분 넘게 비어 있음', '10분 넘게', '1시간 넘게', '1일 넘게'])
gap_tbl['비율(%)'] = (gap_tbl['구간 수'] / len(g) * 100).round(2)
display(gap_tbl.rename_axis('기록 간격'))
print(f"한 분에 기록이 15개 미만인 분: {(df.n_rows < 15).mean() * 100:.1f}%")

big = g[g > 60]
fig = px.histogram(x=np.log10(big), nbins=60, color_discrete_sequence=[C_OK],
                   labels={'x': '비어 있던 시간 (초, 로그10)'}, title='그림 3-1. 1분 넘게 기록이 비었던 구간의 길이 분포')
fig.update_layout(xaxis=dict(tickvals=[2, 3, 4, 5, 6], ticktext=['100초', '17분', '2.8시간', '1.2일', '12일']),
                  yaxis_title='구간 수')
show(fig, 420)

,결측률(%)
컬럼,
recipe_step,0.000
shutter,0.001
TTF_FlowCool Pressure Dropped Below Limit,24.685
TTF_Flowcool Pressure Too High Check Flowcool Pump,31.852
TTF_Flowcool leak,53.999


센서 평균 16개 결측: 0 건


,구간 수,비율(%)
기록 간격,,
1분 넘게 비어 있음,398682,6.90
10분 넘게,68187,1.18
1시간 넘게,22973,0.40
1일 넘게,324,0.01


한 분에 기록이 15개 미만인 분: 11.1%


### 3-2. 평소 기준과 고장 전 라벨

센서 값은 레시피·단계·셔터 조건마다 평소 범위가 다르다. 그래서 **같은 장비·레시피·단계·셔터 조건의 평소 구간**에서 평균·표준편차를 구해 z값으로 바꾼다.

- **평소 구간**: 어떤 고장 기록에서도 앞뒤 24시간 넘게 떨어진 분. 정상으로 확정된 라벨은 아니다.
- **고장 전 24시간**: `TTF ≤ 86,400`(초 가정)인 분. 고장 종류별로 따로 만든다. 다른 고장이 같은 구간에 겹칠 수 있다.
- 조건별 평소 분이 **50개 미만**이거나 표준편차가 1e-6 이하면 z값을 만들지 않는다(결측).
- 고장 시각은 정비로 장비를 세운 시점이라, 실제 이상이 시작된 시점은 더 이를 수 있다.

In [4]:
def nearest_fault_dist(d, ev):
    out = np.full(len(d), np.inf)
    t_all = d.time_start.to_numpy()
    for tool, idx in d.groupby('Tool', observed=True).indices.items():
        ft = np.sort(ev.loc[ev.Tool == tool, 'time'].unique())
        if len(ft) == 0:
            continue
        t = t_all[idx]
        j = np.searchsorted(ft, t)
        prev = np.where(j > 0, t - ft[np.maximum(j - 1, 0)], np.inf)
        nxt = np.where(j < len(ft), ft[np.minimum(j, len(ft) - 1)] - t, np.inf)
        out[idx] = np.minimum(prev, nxt)
    return out

df['fault_dist'] = nearest_fault_dist(df, events)
normal = (df.fault_dist > 86400).to_numpy()

df['_rs'] = df.recipe_step.fillna(-1)
df['_sh'] = df.shutter.fillna(-1)
df['ctx'] = df.groupby(['Tool', 'recipe', '_rs', '_sh'], observed=True, sort=False).ngroup()
nb = df.loc[normal]
cnt = nb.groupby('ctx').size()
st = nb.groupby('ctx')[FEAT].agg(['mean', 'std'])
valid = cnt.ge(50)
ctx_arr = df.ctx.to_numpy()
Z = pd.DataFrame(index=df.index)
for s, f in zip(SENSORS, FEAT):
    m = st[(f, 'mean')].reindex(ctx_arr).to_numpy()
    sd = st[(f, 'std')].where(valid & (st[(f, 'std')] > 1e-6)).reindex(ctx_arr).to_numpy()
    Z[s] = ((df[f].to_numpy() - m) / sd).astype('float32')

pre = {f: (df[TTF[f]] <= 86400).to_numpy() for f in FAULTS}
rows = [{'구분': '평소 (고장에서 24시간 넘게 떨어짐)', '분 수': int(normal.sum()),
         '냉각 압력 z 있음(%)': round(Z.FLOWCOOLPRESSURE[normal].notna().mean() * 100, 1)}]
for f in FAULTS:
    rows.append({'구분': f'{FSHORT[f]} 24시간 전', '분 수': int(pre[f].sum()),
                 '냉각 압력 z 있음(%)': round(Z.FLOWCOOLPRESSURE[pre[f]].notna().mean() * 100, 1)})
display(pd.DataFrame(rows).set_index('구분'))
print(f'조건(장비·레시피·단계·셔터) 조합 {df.ctx.nunique():,}개 중 평소 분 50개 이상: {int(valid.sum()):,}개')

,분 수,냉각 압력 z 있음(%)
구분,,
평소 (고장에서 24시간 넘게 떨어짐),5032183,98.3
압력 저하 24시간 전,347456,95.8
압력 과다 24시간 전,98989,90.6
누설 24시간 전,48977,95.9


조건(장비·레시피·단계·셔터) 조합 11,370개 중 평소 분 50개 이상: 2,614개


## 4. 데이터 시각화·상세 분석

### 4-1. 장비별 고장 분포 (Q3)

고장 건수가 장비에 몰려 있으면, 학습·검증에 같은 장비가 섞일 때 점수가 부풀려진다. 장비 단위로 나눠 평가해야 하는 이유다.

In [5]:
tot = fc.sum(axis=1).sort_values(ascending=False)
share = (tot / tot.sum() * 100).round(1)
display(pd.DataFrame({'고장 기록': tot, '전체 대비(%)': share, '누적(%)': share.cumsum().round(1)}).head(8))

fig = px.bar(fc.loc[tot.index].reset_index().melt(id_vars='Tool', var_name='고장 종류', value_name='건수'),
             x='Tool', y='건수', color='고장 종류', barmode='stack',
             color_discrete_sequence=['#2a78d6', '#eb6834', '#7a7f87'],
             title='그림 4-1. 장비별 고장 기록 수 — 일부 장비에 쏠려 있다')
show(fig, 450)
top = tot.index[0]
print(f'가장 많은 {top}: {tot.iloc[0]}건({share.iloc[0]}%), 가장 적은 {tot.index[-1]}: {tot.iloc[-1]}건')

,고장 기록,전체 대비(%),누적(%)
Tool,,,
03M02,356,29.4,29.4
09M01,150,12.4,41.8
08M01,117,9.7,51.5
01M02,109,9.0,60.5
03M01,91,7.5,68.0
09M02,65,5.4,73.4
06M02,59,4.9,78.3
04M02,37,3.1,81.4


가장 많은 03M02: 356건(29.4%), 가장 적은 10M02: 2건


### 4-2. 센서 분포와 장비 간 차이 (Q3)

센서는 20대를 합쳐 표준화돼 있어서 **장비별 평균이 0에서 얼마나 벗어나는지**가 곧 장비 간 차이다. 극단값은 지우지 않고 보고만 한다. 냉각 압력의 튀는 값은 고장의 전조일 수도 있어서다.

In [6]:
tool_mean = df.groupby('Tool', observed=True)[FEAT].mean()
tool_mean.columns = [NAMES[s] for s in SENSORS]
fig = px.imshow(tool_mean, zmin=-1.5, zmax=1.5, color_continuous_scale='RdBu_r', aspect='auto',
                labels={'color': '장비 평균 (표준화 값)'}, title='그림 4-2. 장비별 센서 평균 — 장비마다 평소 수준이 다르다')
fig.update_xaxes(tickangle=-40)
show(fig, 640)

x = df[FEAT]
desc = pd.DataFrame({'평균': x.mean(), '표준편차': x.std(), '최소': x.min(), '최대': x.max(),
                     '|값|>5 비율(%)': (x.abs() > 5).mean() * 100,
                     '장비 평균의 최대 이탈': tool_mean.abs().max().to_numpy()})
desc.index = [NAMES[s] for s in SENSORS]
display(desc.round(3))

,평균,표준편차,최소,최대,|값|>5 비율(%),장비 평균의 최대 이탈
챔버 압력,-0.092,0.951,-1.908000,99.363998,0.008,0.387
빔 전압,-0.067,0.986,-1.529000,3.696000,0.000,0.155
빔 전류,-0.070,0.979,-1.519000,3.713000,0.000,0.153
억제 전압,-0.079,0.987,-1.575000,2.033000,0.000,0.155
억제 전류,-0.071,0.991,-1.515000,24.652000,0.003,0.206
냉각 유량,-0.106,1.016,-3.483000,1.406000,0.000,0.183
냉각 압력,-0.067,1.101,-4.230000,16.629999,0.601,0.162
아르곤 유량(소스),-0.086,0.991,-1.901000,2.699000,0.000,0.173
아르곤 유량(PBN),-0.097,1.007,-3.826000,2.329000,0.000,0.176
웨이퍼 기울기,0.088,1.002,-1.799000,22.285999,0.049,0.152


### 4-3. 센서 간 중복 (Q5)

1분 평균 577만 행에서 고정 시드로 30만 행을 뽑아 Pearson 상관을 계산했다(상관만 표본, 나머지는 전체). 상관이 높다고 같은 센서라는 뜻은 아니다.

In [7]:
samp = df[FEAT].sample(300_000, random_state=0)
samp.columns = [NAMES[s] for s in SENSORS]
corr = samp.corr()
fig = px.imshow(corr, zmin=-1, zmax=1, color_continuous_scale='RdBu_r', aspect='auto',
                labels={'color': 'Pearson r'}, title='그림 4-3. 센서 간 상관 (30만 행 표본)')
fig.update_xaxes(tickangle=-40)
show(fig, 640)

pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().rename('r').reset_index()
pairs.columns = ['센서 A', '센서 B', 'r']
display(pairs.reindex(pairs.r.abs().sort_values(ascending=False).index).head(8).round(3).reset_index(drop=True))

,센서 A,센서 B,r
0,빔 전류,억제 전류,0.977
1,빔 전압,빔 전류,0.970
2,빔 전압,억제 전류,0.937
3,냉각 유량,아르곤 유량(PBN),0.933
4,아르곤 유량(소스),아르곤 유량(PBN),0.901
5,챔버 압력,아르곤 유량(소스),0.882
6,챔버 압력,아르곤 유량(PBN),0.869
7,억제 전류,아르곤 유량(소스),0.863


### 4-4. 고장 직전에 관측이 있는가 (Q4)

고장 시각은 장비를 세운 시점이라, 직전에 이미 멈춰 있으면 **기록이 없다**. 각 고장 기록에서 직전 1시간·24시간 동안 기록이 있는 분의 비율(점유율)을 센다. 관측이 없다고 돌발 고장이라고 분류하지 않는다.

In [8]:
t_all = df.time_start.to_numpy()
cov_rows = []
for tool, idx in df.groupby('Tool', observed=True).indices.items():
    t = t_all[idx]
    for e in events[events.Tool == tool].itertuples():
        hi = np.searchsorted(t, e.time, 'left')
        n1 = hi - np.searchsorted(t, e.time - HOUR, 'left')
        n24 = hi - np.searchsorted(t, e.time - 24 * HOUR, 'left')
        cov_rows.append((tool, e.fault_type, e.time, n1 / 60, n24 / 1440))
cov = pd.DataFrame(cov_rows, columns=['Tool', 'fault_type', 'time', 'occ_1h', 'occ_24h'])
cov['고장 종류'] = cov.fault_type.map(FSHORT)

summ = cov.groupby('고장 종류').agg(고장_기록=('time', 'size'),
                                 직전1시간_기록없음=('occ_1h', lambda s: (s == 0).sum()),
                                 직전24시간_점유율_중앙값=('occ_24h', 'median'),
                                 점유율_50퍼센트_이상=('occ_24h', lambda s: (s >= .5).sum()))
summ['직전1시간 기록없음(%)'] = (summ.직전1시간_기록없음 / summ.고장_기록 * 100).round(1)
summ['직전24시간_점유율_중앙값'] = (summ.직전24시간_점유율_중앙값 * 100).round(1)
display(summ.loc[[FSHORT[f] for f in FAULTS]])

fig = px.histogram(cov, x=cov.occ_24h * 100, color='고장 종류', facet_col='고장 종류', nbins=25,
                   color_discrete_sequence=['#2a78d6', '#eb6834', '#7a7f87'],
                   labels={'x': '직전 24시간 중 기록이 있는 분의 비율(%)'},
                   title='그림 4-4. 고장 직전 24시간 관측 점유율 — 고장마다 크게 다르다')
fig.update_layout(showlegend=False, yaxis_title='고장 기록 수')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
show(fig, 420)

,고장_기록,직전1시간_기록없음,직전24시간_점유율_중앙값,점유율_50퍼센트_이상,직전1시간 기록없음(%)
고장 종류,,,,,
압력 저하,741,235,49.8,365,31.7
압력 과다,333,201,27.8,83,60.4
누설,137,41,41.6,62,29.9


### 4-5. 센서별 고장 전·평소 구분력 (Q1)

각 장비 안에서 **고장 24시간 전 분의 z값**과 **평소 분의 z값**(장비당 최대 5만 분 표본)을 비교한다. AUC가 0.5면 구분이 안 되고, 0.5에서 멀수록 값의 순위가 다르다. 방향은 신호가 아니라 “고장 전에 평소보다 크면 0.5 초과”를 뜻한다.

장비마다 따로 계산해 **중앙값**으로 요약한다(고장 전 분 200개 이상인 장비만). 같은 고장의 연속된 분은 독립이 아니며, 분류 성능이 아니라 순위 차이의 크기다.

In [9]:
rng = np.random.default_rng(0)
tool_idx = df.groupby('Tool', observed=True).indices
auc_rows = []
for tool, idx in tool_idx.items():
    nrm_idx = idx[normal[idx]]
    if len(nrm_idx) < 1000:
        continue
    nrm_idx = rng.choice(nrm_idx, size=min(50_000, len(nrm_idx)), replace=False)
    for f in FAULTS:
        pidx = idx[pre[f][idx]]
        if len(pidx) < 200:
            continue
        for s in SENSORS:
            x1 = Z[s].to_numpy()[pidx]; x0 = Z[s].to_numpy()[nrm_idx]
            x1 = x1[~np.isnan(x1)]; x0 = x0[~np.isnan(x0)]
            if len(x1) < 200 or len(x0) < 1000:
                continue
            r = rankdata(np.concatenate([x1, x0]))
            auc = (r[:len(x1)].sum() - len(x1) * (len(x1) + 1) / 2) / (len(x1) * len(x0))
            auc_rows.append((tool, FSHORT[f], s, auc, len(x1)))
auc = pd.DataFrame(auc_rows, columns=['Tool', '고장 종류', 'sensor', 'auc', 'n_pre'])
auc_med = auc.pivot_table(index='sensor', columns='고장 종류', values='auc', aggfunc='median').reindex(index=SENSORS)
auc_n = auc.pivot_table(index='sensor', columns='고장 종류', values='Tool', aggfunc='nunique').reindex(index=SENSORS)
order = [FSHORT[f] for f in FAULTS]

hm = auc_med[order].copy(); hm.index = [NAMES[s] for s in SENSORS]
fig = px.imshow(hm, zmin=.2, zmax=.8, color_continuous_scale='RdBu_r', aspect='auto', text_auto='.2f',
                labels={'color': '장비별 AUC 중앙값'}, title='그림 4-5. 고장 24시간 전 vs 평소 — 센서별 AUC (0.5 = 구분 안 됨)')
show(fig, 640)
print('비교에 쓴 장비 수:', {k: int(auc[auc['고장 종류'] == k].Tool.nunique()) for k in order})

top_rows = []
for k in order:
    d = (auc_med[k] - .5).abs().sort_values(ascending=False).head(3)
    for s in d.index:
        top_rows.append({'고장 종류': k, '센서': NAMES[s], 'AUC 중앙값': round(auc_med.loc[s, k], 3), '장비 수': int(auc_n.loc[s, k])})
display(pd.DataFrame(top_rows))

비교에 쓴 장비 수: {'압력 저하': 19, '압력 과다': 18, '누설': 14}


,고장 종류,센서,AUC 중앙값,장비 수
0,압력 저하,실드 사용량2,0.598,19
1,압력 저하,회전 속도,0.455,13
2,압력 저하,냉각 압력,0.466,19
3,압력 과다,실드 사용량1,0.422,18
4,압력 과다,아르곤 유량(PBN),0.458,18
5,압력 과다,챔버 압력,0.459,18
6,누설,실드 사용량1,0.339,14
7,누설,실드 사용량2,0.422,14
8,누설,냉각 압력,0.438,14


### 4-6. 고장 전 궤적 (Q2)

고장 기록 1,211건마다 **고장 전 48시간**을 1시간 단위로 나눠 센서 z값의 평균을 구하고, 사건 사이의 **중앙값과 사분위 범위**를 그린다. 0은 같은 조건의 평소 수준이다. 기록이 있는 사건만 시간별 분모에 들어가서 시간마다 사건 수가 다르다. 같은 장비의 반복 고장은 독립이 아니다.

In [10]:
KEY = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE', 'IONGAUGEPRESSURE']
H = 48
traj_parts = []
zk = Z[KEY].to_numpy()
for tool, idx in tool_idx.items():
    t = t_all[idx]; zz = zk[idx]
    for e in events[events.Tool == tool].itertuples():
        lo = np.searchsorted(t, e.time - H * HOUR, 'left'); hi = np.searchsorted(t, e.time, 'left')
        if hi <= lo:
            continue
        blk = pd.DataFrame(zz[lo:hi], columns=KEY)
        blk['h'] = (e.time - t[lo:hi] - 1) // HOUR
        g = blk.groupby('h')[KEY].mean().reset_index()
        g['event'] = f'{tool}_{e.time}_{e.fault_type}'; g['fault_type'] = e.fault_type
        traj_parts.append(g)
traj = pd.concat(traj_parts, ignore_index=True)
prof = (traj.melt(id_vars=['event', 'fault_type', 'h'], value_vars=KEY, var_name='sensor', value_name='z').dropna()
        .groupby(['fault_type', 'sensor', 'h']).z
        .agg(median='median', q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75), n_events='size').reset_index())

fig = make_subplots(rows=len(KEY), cols=3, shared_xaxes=True, vertical_spacing=.07,
                    subplot_titles=[f'{NAMES[s]} · {FSHORT[f]}' for s in KEY for f in FAULTS])
colors = {'FLOWCOOLPRESSURE': '#2a78d6', 'FLOWCOOLFLOWRATE': '#eb6834', 'IONGAUGEPRESSURE': '#4c9a52'}
for i, s in enumerate(KEY, 1):
    for j, f in enumerate(FAULTS, 1):
        p = prof[(prof.sensor == s) & (prof.fault_type == f)].sort_values('h')
        x = -p.h.to_numpy() - 0.5
        fig.add_scatter(x=np.r_[x, x[::-1]], y=np.r_[p.q75, p.q25[::-1]], fill='toself', mode='none',
                        fillcolor='rgba(120,120,120,.18)', showlegend=False, row=i, col=j)
        fig.add_scatter(x=x, y=p['median'], mode='lines', line=dict(color=colors[s], width=2), showlegend=False,
                        hovertemplate='%{x:.1f}시간 전<br>중앙값 %{y:.3f}<extra></extra>', row=i, col=j)
        fig.add_hline(y=0, line=dict(color='#999', dash='dot', width=1), row=i, col=j)
fig.update_xaxes(title_text='고장까지 남은 시간(시간)', row=len(KEY))
fig.update_layout(title='그림 4-6. 고장 전 48시간 궤적 — 사건 간 중앙값(선)과 사분위 범위(띠)')
show(fig, 800)

late = prof[prof.h < 3].groupby(['fault_type', 'sensor'])['median'].median()
early = prof[(prof.h >= 24) & (prof.h < 48)].groupby(['fault_type', 'sensor'])['median'].median()
delta = (late - early).rename('직전 3시간 − 24~48시간 전').reset_index()
delta['고장 종류'] = delta.fault_type.map(FSHORT); delta['센서'] = delta.sensor.map(NAMES)
nev = traj.groupby('fault_type').event.nunique()
delta['기록 있는 사건'] = delta.fault_type.map(nev)
display(delta[['고장 종류', '센서', '직전 3시간 − 24~48시간 전', '기록 있는 사건']].round(3).reset_index(drop=True))

,고장 종류,센서,직전 3시간 − 24~48시간 전,기록 있는 사건
0,압력 저하,냉각 유량,-0.015,739
1,압력 저하,냉각 압력,0.052,739
2,압력 저하,챔버 압력,-0.061,739
3,압력 과다,냉각 유량,-0.032,270
4,압력 과다,냉각 압력,0.052,270
5,압력 과다,챔버 압력,-0.118,270
6,누설,냉각 유량,-0.073,133
7,누설,냉각 압력,-0.144,133
8,누설,챔버 압력,-0.115,133


### 4-7. 고장 기록을 사건으로 묶는 민감도 (Q4)

같은 고장이 몇 시간 간격으로 반복 기록된다. 같은 장비·종류에서 **앞 기록과의 간격이 기준보다 길 때만 새 사건**으로 센다. 이는 실제 사건의 정답이 아니라 민감도 분석이고, 원래 고장 기록과 TTF는 바꾸지 않았다.

In [11]:
ep_rows = []
for gap, label in [(0, '묶지 않음'), (3600, '1시간'), (21600, '6시간'), (86400, '1일')]:
    ev = events.sort_values(['Tool', 'fault_type', 'time']).copy()
    d = ev.groupby(['Tool', 'fault_type']).time.diff()
    ev['new'] = d.isna() | (d > gap)
    row = {'묶는 간격': label}
    for f in FAULTS:
        row[FSHORT[f]] = int(ev.loc[ev.fault_type == f, 'new'].sum())
    row['합계'] = int(ev.new.sum())
    ep_rows.append(row)
ep = pd.DataFrame(ep_rows).set_index('묶는 간격')
display(ep)

fig = px.bar(ep.reset_index().melt(id_vars='묶는 간격', value_vars=order, var_name='고장 종류', value_name='사건 수'),
             x='묶는 간격', y='사건 수', color='고장 종류', barmode='group',
             color_discrete_sequence=['#2a78d6', '#eb6834', '#7a7f87'],
             title='그림 4-7. 묶는 간격에 따른 사건 수 — 기록 1,211건이 1일 기준으로 줄어든다')
show(fig, 420)

,압력 저하,압력 과다,누설,합계
묶는 간격,,,,
묶지 않음,741,333,137,1211
1시간,701,278,112,1091
6시간,582,245,77,904
1일,383,208,57,648


## 5. 인사이트 → 모델링 반영

### 5-1. 실제 결과 요약

In [12]:
none1 = (cov.occ_1h == 0).mean() * 100
best = {k: (auc_med[k] - .5).abs().idxmax() for k in order}
best_txt = ', '.join(f"{k}: **{NAMES[best[k]]}**(AUC {auc_med.loc[best[k], k]:.2f})" for k in order)
pr = delta[delta.sensor == 'FLOWCOOLPRESSURE'].set_index('fault_type')['직전 3시간 − 24~48시간 전']
pr_txt = ', '.join(f"{FSHORT[f]} {pr[f]:+.2f}" for f in FAULTS)
display(Markdown(f"""
- **대상·기간**: 장비 20대 전체 기간, 1분 요약 **{len(df):,}행**. 고장 기록 {len(faults):,}행(중복 제외 {len(events):,}건)이며 1일 기준으로 묶으면 **{ep.loc['1일', '합계']}건**이다.
- **품질**: 센서 값 결측은 없다. 1시간 넘게 기록이 빈 구간이 {gap_tbl.loc['1시간 넘게', '구간 수']:,}곳이고, 한 분에 기록이 15개 미만인 분이 {(df.n_rows < 15).mean() * 100:.1f}%다.
- **장비 간 차이**: 고장이 가장 많은 {tot.index[0]}이 전체의 {share.iloc[0]}%, 상위 3대가 {share.head(3).sum():.1f}%다. 센서 평균도 장비마다 최대 {tool_mean.abs().max().max():.2f}까지 벗어난다.
- **고장 직전 관측**: 고장 기록의 **{none1:.0f}%**는 직전 1시간에 기록이 하나도 없다(장비가 이미 멈춰 있음).
- **센서별 구분력(고장 24시간 전 vs 평소)**: 고장 종류별로 0.5에서 가장 먼 센서는 {best_txt}. 이 중 실드·그리드 **사용량 카운터는 누적값이라 나이 효과일 수 있어** 전조로 단정하지 않는다(5-2 참고).
- **냉각 압력의 직전 변화**(직전 3시간 − 24~48시간 전 z 중앙값): {pr_txt}.
"""))


- **대상·기간**: 장비 20대 전체 기간, 1분 요약 **5,777,042행**. 고장 기록 1,236행(중복 제외 1,211건)이며 1일 기준으로 묶으면 **648건**이다.
- **품질**: 센서 값 결측은 없다. 1시간 넘게 기록이 빈 구간이 22,973곳이고, 한 분에 기록이 15개 미만인 분이 11.1%다.
- **장비 간 차이**: 고장이 가장 많은 03M02이 전체의 29.4%, 상위 3대가 51.5%다. 센서 평균도 장비마다 최대 0.71까지 벗어난다.
- **고장 직전 관측**: 고장 기록의 **39%**는 직전 1시간에 기록이 하나도 없다(장비가 이미 멈춰 있음).
- **센서별 구분력(고장 24시간 전 vs 평소)**: 고장 종류별로 0.5에서 가장 먼 센서는 압력 저하: **실드 사용량2**(AUC 0.60), 압력 과다: **실드 사용량1**(AUC 0.42), 누설: **실드 사용량1**(AUC 0.34). 이 중 실드·그리드 **사용량 카운터는 누적값이라 나이 효과일 수 있어** 전조로 단정하지 않는다(5-2 참고).
- **냉각 압력의 직전 변화**(직전 3시간 − 24~48시간 전 z 중앙값): 압력 저하 +0.05, 압력 과다 +0.05, 누설 -0.14.


### 5-2. 컬럼 후보 결정표

고장 3종에서 구분력(0.5에서 떨어진 정도) 중 **가장 큰 값**을 기준으로 줄 세운 **초안**이다. 학습 성능으로 정한 최종 목록이 아니다. 기준: 0.10 이상 *우선 후보*, 0.05~0.10 *조건부*, 0.05 미만 *보류*.

**주의**: `ETCHSOURCEUSAGE`·`ETCHAUXSOURCETIMER`·`ETCHAUX2SOURCETIMER`는 소모품을 쓴 양을 **누적하는 카운터**다. 정비로 장비를 세우면 리셋되는 구조라면, 고장(=정비 직전)에는 값이 크고 평소에는 작아서 **고장 전조가 아니어도 구분력이 높게 나온다.** 카운터가 언제 리셋되는지 확인하기 전에는 이 3개의 순위를 전조 신호로 해석하지 않는다. 나머지 13개의 구분력은 대체로 0.5에 가깝다(0.46~0.53).

In [13]:
dev = (auc_med[order] - .5).abs()
dec = pd.DataFrame({'센서': [NAMES[s] for s in SENSORS], '컬럼': SENSORS,
                    '압력 저하 AUC': auc_med[order[0]].round(3).to_numpy(),
                    '압력 과다 AUC': auc_med[order[1]].round(3).to_numpy(),
                    '누설 AUC': auc_med[order[2]].round(3).to_numpy(),
                    '최대 이탈': dev.max(axis=1).round(3).to_numpy()})
dec['판정(초안)'] = np.where(dec['최대 이탈'] >= .10, '우선 후보', np.where(dec['최대 이탈'] >= .05, '조건부', '보류'))
counter = {'ETCHSOURCEUSAGE', 'ETCHAUXSOURCETIMER', 'ETCHAUX2SOURCETIMER'}
dec['비고'] = np.where(dec['컬럼'].isin(counter), '누적 카운터: 고장 신호가 아니라 사용 시점(나이) 효과일 수 있음', '')
display(dec.sort_values('최대 이탈', ascending=False).reset_index(drop=True))

,센서,컬럼,압력 저하 AUC,압력 과다 AUC,누설 AUC,최대 이탈,판정(초안),비고
0,실드 사용량1,ETCHAUXSOURCETIMER,0.504,0.422,0.339,0.161,우선 후보,누적 카운터: 고장 신호가 아니라 사용 시점(나이) 효과일 수 있음
1,실드 사용량2,ETCHAUX2SOURCETIMER,0.598,0.493,0.422,0.098,조건부,누적 카운터: 고장 신호가 아니라 사용 시점(나이) 효과일 수 있음
2,냉각 압력,FLOWCOOLPRESSURE,0.466,0.524,0.438,0.062,조건부,
3,억제 전압,ETCHSUPPRESSORVOLTAGE,0.472,0.459,0.454,0.046,보류,
4,회전 속도,ROTATIONSPEED,0.455,0.480,0.471,0.045,보류,
5,아르곤 유량(PBN),ETCHPBNGASREADBACK,0.493,0.458,0.464,0.042,보류,
6,챔버 압력,IONGAUGEPRESSURE,0.475,0.459,0.479,0.041,보류,
7,빔 전압,ETCHBEAMVOLTAGE,0.467,0.465,0.462,0.038,보류,
8,빔 전류,ETCHBEAMCURRENT,0.471,0.467,0.473,0.033,보류,
9,아르곤 유량(소스),ETCHGASCHANNEL1READBACK,0.487,0.480,0.468,0.032,보류,


### 5-3. 적용 범위·한계·다음 단계

- 이 결과는 **1분 평균**이다. 1분보다 짧은 순간 변화(스파이크, 4초 해상도)는 보이지 않는다. 그런 분석은 `data/processed/prefault/`의 원본 행으로 한다.
- 센서는 표준화돼 있어 **실제 단위의 관리 기준은 만들 수 없다.** 평소 대비 이탈로 경보 기준을 정한다.
- 고장 시각은 정비 정지 시각이다. 실제 이상 시작은 더 이를 수 있고 제공되지 않는다. 직전에 기록이 없는 고장은 전조를 볼 수 없다.
- ‘평소’는 고장에서 24시간 넘게 떨어진 구간일 뿐 **정상으로 확정된 라벨이 아니다.** 레시피가 평소와 달라진 효과를 완전히 통제하지 못한다.
- 같은 고장이 반복 기록되고 같은 장비의 관측은 독립이 아니다. 사건 단위 규칙(몇 시간·며칠로 묶을지)을 정하기 전의 탐색 결과다.
- 장비 간 차이가 커서 검증은 **장비 단위로 나눠야** 한다. 장비별 평소 기준(레시피별)을 학습 구간에서만 계산해야 미래 정보가 새지 않는다.
- 이 EDA는 예측 가능성을 입증하지 않는다. 다음 단계는 사건 정의와 관측 종료를 반영한 라벨링, 시간순·장비 단위 검증, 과거 정보만 쓰는 피처 설계이며, 예측 구간과 탐지율·오경보·사전 경고 시간을 함께 정한다.
- 시험 장비 5대는 정답이 공개되지 않아 쓰지 않았다.

### 재현 경로

1. 원본을 `data/raw/`에 두고 압축을 푼다(루트 `README.md` 참고).
2. `src/make_processed.py`: 장비별 1분 요약(`data/processed/minute/`)과 고장 24시간 전 원본 구간, `faults.csv`를 만든다.
3. 이 노트북을 처음부터 실행한다. 그림·표는 파일로 저장하지 않고 노트북 출력에만 남는다.